## Libraries and data

### Libraries

In [1]:
import networkx as nx
import pandas as pd

from pyvis.network import Network
from time import time

from semanticscholar import SemanticScholar

import matplotlib.pyplot as plt



### Data

In [2]:
# Load dataframe

DIR = "/home/p0l3/RAD/BERT_PRETRAINING/PRETRAINING/DATASET/ED4RE_2503/ED4RE_2603.pickle"
df = pd.read_pickle(DIR)

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 171880 entries, 0 to 10
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   Title                     171880 non-null  object
 1   Authors_and_Affiliations  171880 non-null  object
 2   Affiliations              171880 non-null  object
 3   DOI                       171880 non-null  object
 4   Authors                   171880 non-null  object
 5   Journal                   171880 non-null  object
 6   Date                      171880 non-null  object
 7   Subjects                  171880 non-null  object
 8   Abstract                  171880 non-null  object
 9   References                171880 non-null  object
 10  Content                   171880 non-null  object
 11  Keywords                  171880 non-null  object
 12  Style                     171880 non-null  object
dtypes: object(13)
memory usage: 18.4+ MB
None


In [8]:
# Filter no references or no author

df = df[df["References"] != "no_references"]
df = df[df["Authors"] != "no_author"]

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 828 entries, 0 to 10
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Title                     828 non-null    object
 1   Authors_and_Affiliations  828 non-null    object
 2   Affiliations              828 non-null    object
 3   DOI                       828 non-null    object
 4   Authors                   828 non-null    object
 5   Journal                   828 non-null    object
 6   Date                      828 non-null    object
 7   Subjects                  828 non-null    object
 8   Abstract                  828 non-null    object
 9   References                828 non-null    object
 10  Content                   828 non-null    object
 11  Keywords                  828 non-null    object
 12  Style                     828 non-null    int64 
dtypes: int64(1), object(12)
memory usage: 90.6+ KB


> **Observation 1**: 96.84 % of data is kept after filtering no_author/no_references. -> NOTE: This is true for Energy Policy dataframe and doesn't consider wrongnly extracted strings.

In [48]:
def preprocess_doi(doi_string):
    if type(doi_string) == type(["List"]):
        doi_string = doi_string[0]
    if doi_string[0] == "/":
        return doi_string[1:]
    return doi_string

In [46]:
# Create list of DOIs

DOI_list = df["DOI"].tolist()
print(len(DOI_list[::1000]))

172


### Helper functions

In [20]:
# Helper functions

def pyvis_show(graf, phi=False, name="Default"):
    net = Network()
    net.from_nx(graf)
    net.toggle_physics(phi)
    net.show(f"{name}_{time()}.html", notebook=False)

def networkx_plot(graf, name="Default"):
    node_sizes = [50*len(graf.edges(n)) for n in graf.nodes()]
    edges = graf.edges()
    #weights = [graf[u][v]['weight']/100 for u,v in edges]
    nx.draw_networkx(graf, pos=nx.kamada_kawai_layout(graf), node_size=node_sizes)
    plt.show() 

## Semantic Scholar Querying

In [ ]:
sch = SemanticScholar()

list_of_paper_ids = [preprocess_doi(doi_string) for doi_string in DOI_list[::1000]]
print(list_of_paper_ids)
results = sch.get_papers(list_of_paper_ids) # 500 papers per query is maximum -> 400 queries for all data

print(results)

Found 171880 valid DOIs to process.
Querying Semantic Scholar in 18 chunks...


Fetching paper data:   0%|          | 0/18 [00:00<?, ?it/s]

['10.1002/joc.3724', '10.1002/joc.4400', '10.1002/joc.7908', '10.1002/joc.7361', 'Climate 2022, 10(4), 49; https://doi.org/10.3390/cli10040049', 'Forests 2019, 10(1), 54; https://doi.org/10.3390/f10010054', 'Atmosphere 2022, 13(7), 1042; https://doi.org/10.3390/atmos13071042', 'Water 2021, 13(19), 2660; https://doi.org/10.3390/w13192660', 'Forests 2015, 6(2), 433-449; https://doi.org/10.3390/f6020433', 'Energies 2016, 9(9), 746; https://doi.org/10.3390/en9090746']
An error occurred in a chunk: 'NoneType' object is not iterable
['Water 2022, 14(14), 2146; https://doi.org/10.3390/w14142146', 'Water 2023, 15(20), 3654; https://doi.org/10.3390/w15203654', 'Forests 2020, 11(12), 1269; https://doi.org/10.3390/f11121269', 'Water 2021, 13(7), 927; https://doi.org/10.3390/w13070927', 'Forests 2022, 13(6), 901; https://doi.org/10.3390/f13060901', 'Water 2020, 12(3), 769; https://doi.org/10.3390/w12030769', 'Water 2017, 9(9), 643; https://doi.org/10.3390/w9090643', 'Forests 2018, 9(5), 255; https

KeyboardInterrupt: 

## Graph

In [42]:
# Create a graph
G = nx.Graph()
N = 500
for k, item in enumerate(results):
    

    authors = [(author['authorId'], author['name']) for author in item.authors]

    for i, author1 in enumerate(authors):
        for j, author2 in enumerate(authors):
            if i != j:
                G.add_node(author1[0], label=author1[1])
                G.add_node(author2[0], label=author2[1])
                G.add_edge(author1[0], author2[0])       
    if k > N:
        break
# Plot the graph
pyvis_show(G, phi=True, name="author_reference_graph")

ValueError: None cannot be a node